* 학년: 
* 반:
* 번호:
* 이름:


# 선택 활동 5: 항공사의 고객 만족도를 높이기 위한 고객 서비스 모델 만들기

**문제 상황** 고객들은 비행기를 선택할 때 어떤 항공사를 선택할까? 고객의 만족도를 높이기 위해 항공사는 어떤 서비스에 중점을 두어야 할까? 고객 항공사 만족도 조사 데이터로 기계학습 모델을 만들어 중점을 두어야 하는 고객 서비스를 알아보자.


**단계별 처리 과정**

| 단계 | 과정 | 처리 내용 |
|:---:|---|---|
| 단계 1 | 문제 정의하기 | 항공사는 고객의 만족도를 높이기 위해 어떤 서비스에 중점을 두어야 할까? |
| 단계 2 | 데이터 수집 및 전처리하기 | 캐글에서 데이터 수집하기: `airline_passenger_satisfaction.csv`<br>특징과 타깃 선정하기<br>데이터 분할하기 |
| 단계 3 | 모델 생성하기 | 결정트리, 로지스틱 회귀 모델 생성하기 |
| 단계 4 | 모델 평가하기 | 성능 평가하기 |


## 단계 0: 준비 (라이브러리 설치)


In [ ]:
import importlib, sys, subprocess

packages = [
    ('pandas', 'pandas'),
    ('numpy', 'numpy'),
    ('sklearn', 'scikit-learn'),
    ('matplotlib.pyplot', 'matplotlib'),
    ('seaborn', 'seaborn'),
]

for module_name, pip_name in packages:
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

from sklearn import set_config
set_config(display="text")

print('라이브러리 준비 완료')

## 단계 1: 문제 정의하기

항공사는 고객의 만족도를 높이기 위해 어떤 서비스에 중점을 두어야 할까?


## 단계 2: 데이터 수집 및 전처리하기

### (1) 데이터 수집하기

캐글 사이트에서 항공사 만족도 데이터인 `airline_passenger_satisfaction.csv` 파일을 다운로드한다.

https://www.kaggle.com/datasets/mysarahmadbhat/airline-passenger-satisfaction

### (2) 데이터 전처리하기


① `airline_passenger_satisfaction.csv` 파일을 불러온다.

② 분류 모델에 사용할 특징과 타깃을 선정한다.

> 9~22번 인덱스 컬럼을 특징(`X`)으로, 마지막 `Satisfaction` 컬럼은 타깃(`y`)으로 선정한다.

1. `pandas` 패키지를 불러온다.
2. `airline_passenger_satisfaction.csv` 파일을 불러와 데이터 프레임 `df`를 생성한다.
3. `iloc`로 모든 행의 9~22번 인덱스 열을 특징 `X`로 선정한다.
4. `iloc`로 모든 행의 마지막 열을 타깃 `y`로 선정한다.
5. 특징과 타깃 데이터의 모양을 출력한다.


In [ ]:
# 특징: 모든 행의 9~22번 인덱스 열
# 타깃: 모든 행의 마지막 열(-1)

| 구분 | 인덱스 | 만족도 질문 | 의미 |
|---|---:|---|---|
| 특징(X) | 9 | Departure and Arrival Time Convenience | 출발, 도착 시간 편의 |
| 특징(X) | 10 | Ease of Online Booking | 온라인 예약의 용이성 |
| 특징(X) | 11 | Check-in Service | 체크인 서비스 |
| 특징(X) | 12 | Online Boarding | 온라인 탑승 |
| 특징(X) | 13 | Gate Location | 게이트 위치 |
| 특징(X) | 14 | On-board Service | 온 보드 서비스 |
| 특징(X) | 15 | Seat Comfort | 의자의 편안함 |
| 특징(X) | 16 | Leg Room Service | 레그 룸 서비스 |
| 특징(X) | 17 | Cleanliness | 청결 |
| 특징(X) | 18 | Food and Drink | 음식과 음료 |
| 특징(X) | 19 | In-flight Service | 기내 서비스 |
| 특징(X) | 20 | In-flight Wifi Service | 기내 와이파이 서비스 |
| 특징(X) | 21 | In-flight Entertainment | 기내 엔터테인먼트 |
| 특징(X) | 22 | Baggage Handling | 수하물 처리 |
| 타깃(y) | 23 | Satisfaction(Neutral or Dissatisfied/Satisfied) | 만족도(중립 또는 불만족/만족) |

> 항공사 만족도 특징값들은 동일한 데이터 범위이므로, 데이터 정규화가 따로 필요하지 않다.

**❓ 확인하기**

- 고객 만족도를 예측하기 위해 특징 X로 선택한 것은 무엇인가?  → ( )
- 모델이 맞혀야 할 타깃 y는 무엇인가?  → ( )
- 이 문제는 회귀인가, 분류인가? 그 이유는 무엇인가?

  → ( )

③ 특징, 타깃 데이터를 훈련 데이터와 테스트 데이터로 7:3 분할한다.

1. `train_test_split`을 불러온다.
2. `X`와 `y`를 훈련 데이터와 테스트 데이터로 7:3 분할하고, 고객 만족도의 비율이 유지되도록 설정한다.
3. 분할한 데이터를 `X_train`, `X_test`, `y_train`, `y_test`에 차례로 저장한다.


**❓ 확인하기**

- 테스트 데이터는 전체의 몇 %인가?  → ( )
- `stratify=y`로 만족·불만족 고객의 비율을 유지하는 이유는 무엇인가?

  → ( )

## 단계 3: 모델 생성하기

결정트리, 로지스틱 회귀 알고리즘을 선택해 훈련 데이터로 학습시켜 분류 모델을 생성한다.

### 결정트리

1. `sklearn.tree`에서 `DecisionTreeClassifier`를 불러온다.
2. 결정트리 모델을 생성해 `dt`에 저장한다.
3. `fit()`으로 훈련 데이터 `X_train`, `y_train`을 학습시킨다.

### 로지스틱 회귀

4. `sklearn.linear_model`에서 `LogisticRegression`을 불러온다.
5. 로지스틱 회귀 모델을 생성해 `lr`에 저장한다.
6. `fit()`으로 훈련 데이터 `X_train`, `y_train`을 학습시킨다.


**❓ 확인하기**

- 결정트리와 로지스틱 회귀는 어떤 데이터를 이용해 학습하는가?

  → ( )

## 단계 4: 모델 평가하기

테스트 데이터를 이용해 예측을 수행하고 분류 모델의 정확도를 확인한다.

1. `predict()`로 결정트리 모델의 예측 결과를 구해 `dt_pred`에 저장한다.
2. `score()`로 결정트리 모델의 분류 정확도를 출력한다.


In [ ]:
# 결정트리 모델 예측하기

**❓ 확인하기**

- 결정트리의 테스트 정확도는 약 얼마인가?  → ( )
- 테스트 데이터로 예측할 때 사용하는 메서드는 무엇인가?  → ( )
- 분류 모델의 정확도를 확인할 때 사용하는 메서드는 무엇인가?  → ( )

결정트리를 그려 보고 어떤 서비스에 중점을 두는 것이 좋은지 결정트리의 루트 노드의 특징을 찾아보자.

1. 그래프를 그리기 위한 `matplotlib.pyplot`을 `plt`라는 이름으로 불러온다.
2. `sklearn.tree`에서 `plot_tree`를 불러온다.
3. 결정트리를 알아보기 쉽도록 그래프의 크기를 지정한다.
4. 학습한 결정트리 `dt`를 특징 이름과 클래스 이름이 나타나도록 그리고, 노드를 색으로 구분한다.
5. 그래프를 화면에 보여 준다.
6. 결정트리의 첫 번째 분할 특징 번호로 `X`의 열 이름을 찾아 루트 노드의 특징을 출력한다.

   (1) **기본 형식:** 학습한 결정트리의 각 노드에서 사용한 특징 번호는 `모델 변수.tree_.feature`로 확인한다.

   (2) 루트 노드는 첫 번째 노드이므로 뒤에 `[0]`을 붙여 특징 번호를 구한다.

   (3) **기본 형식:** `데이터 프레임 변수.columns[열 번호]`는 해당 번호의 열 이름을 가져온다.

   (4) 데이터 프레임 변수에는 `X`, 열 번호에는 (2)에서 구한 값을 넣는다.


**❓ 확인하기**

- 결정트리의 루트 노드에서 사용한 서비스 특징은 무엇인가?  → ( )
- 루트 노드의 특징은 결정트리가 가장 먼저 고객을 분류할 때 사용한 기준이라는 뜻이다. 항공사가 우선 살펴볼 서비스는 무엇인가?

  → ( )

## 🏁 마무리

1. 결정트리 모델의 정확도는 얼마인가?

   →

2. 결정트리가 가장 중요하게 본 서비스는 무엇인가?

   →

3. 루트 노드의 특징만으로 인과관계를 단정할 수 없는 이유는 무엇인가?

   →